# **Q1(a)**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import random_split, DataLoader

In [ ]:
def train_model(dataset_name, model_name, batch_size, lr, opt, epochs, pin_memory=True, use_amp=True):
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    if dataset_name == "mnist":
        full_dataset = datasets.MNIST("./data", train=True, download=True, transform=transform)
        test_dataset = datasets.MNIST("./data", train=False, transform=transform)

    elif dataset_name == "fashion":
        full_dataset = datasets.FashionMNIST("./data", train=True, download=True, transform=transform)
        test_dataset = datasets.FashionMNIST("./data", train=False, transform=transform)

    else:
        raise ValueError("Unknown dataset")

    train_size = int(0.7 * len(full_dataset))
    val_size = int(0.1 * len(full_dataset))
    _ = len(full_dataset) - train_size - val_size

    train_dataset, val_dataset, _ = random_split(
        full_dataset, [train_size, val_size, _]
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        pin_memory=pin_memory
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        pin_memory=pin_memory
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=1000,
        shuffle=False,
        pin_memory=pin_memory
    )

    if model_name == "resnet18":
        model = models.resnet18(weights=None)
    elif model_name == "resnet50":
        model = models.resnet50(weights=None)
    else:
        raise ValueError("Unknown model")

    model.conv1 = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(model.fc.in_features, 10)

    model.to(DEVICE)


    if opt == "sgd":
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    elif opt == "adam":
        optimizer = optim.Adam(model.parameters(), lr=lr)
    else:
        raise ValueError("Unknown optimizer")

    criterion = nn.CrossEntropyLoss()
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    def validate():
      model.eval()
      correct = 0
      total = 0
      with torch.no_grad():
          for x, y in val_loader:
              x, y = x.to(DEVICE), y.to(DEVICE)
              pred = model(x).argmax(dim=1)
              correct += pred.eq(y).sum().item()
              total += y.size(0)
      return 100.0 * correct / total

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for batch_idx, (x, y) in enumerate(train_loader):
            x, y = x.to(DEVICE), y.to(DEVICE)

            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=use_amp):
                out = model(x)
                loss = criterion(out, y)


            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()
        print(f"Epoch {epoch+1} average loss: {running_loss / len(train_loader):.4f}")

    model.eval()
    correct = 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            pred = model(x).argmax(dim=1)
            correct += pred.eq(y).sum().item()

    test_acc = 100.0 * correct / len(test_dataset)
    return test_acc

MNIST - ResNet-18 - Batch size (16)

In [ ]:
for lr in [0.001, 0.0001]:
  for opt in ["sgd", "adam"]:
        acc = train_model(
            dataset_name="mnist",
            model_name="resnet18",
            batch_size=16,
            lr=lr,
            opt=opt,
            epochs=5,
            pin_memory=True,

            use_amp=True
        )
        print("Dataset : MNIST, Model : ResNet-18")
        print(f"Batch Size : 16, Learning Rate : {lr}, Optimizer : {opt}, Epochs : 5, PIN_MEMORY = True")
        print(f"Test Accuracy : {acc:.2f}%")

/tmp/ipython-input-1142887438.py:95: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/tmp/ipython-input-1142887438.py:122: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


Epoch 1 average loss: 0.2612
Epoch 2 average loss: 0.0471
Epoch 3 average loss: 0.0273
Epoch 4 average loss: 0.0156
Epoch 5 average loss: 0.0083
Dataset : MNIST, Model : ResNet-18
Batch Size : 16, Learning Rate : 0.001, Optimizer : sgd, Epochs : 5, PIN_MEMORY = True
Test Accuracy : 99.34%
Epoch 1 average loss: 0.1439
Epoch 2 average loss: 0.0664
Epoch 3 average loss: 0.0473
Epoch 4 average loss: 0.0348
Epoch 5 average loss: 0.0291
Dataset : MNIST, Model : ResNet-18
Batch Size : 16, Learning Rate : 0.001, Optimizer : adam, Epochs : 5, PIN_MEMORY = True
Test Accuracy : 98.92%
Epoch 1 average loss: 1.2098
Epoch 2 average loss: 0.2920
Epoch 3 average loss: 0.1627
Epoch 4 average loss: 0.1180
Epoch 5 average loss: 0.0917
Dataset : MNIST, Model : ResNet-18
Batch Size : 16, Learning Rate : 0.0001, Optimizer : sgd, Epochs : 5, PIN_MEMORY = True
Test Accuracy : 98.27%
Epoch 1 average loss: 0.1424
Epoch 2 average loss: 0.0452
Epoch 3 average loss: 0.0335
Epoch 4 average loss: 0.0265
Epoch 5 aver

MNIST - ResNet-18 - Batch size (32)

In [ ]:
for lr in [0.001, 0.0001]:
  for opt in ["sgd", "adam"]:
        acc = train_model(
            dataset_name="mnist",
            model_name="resnet18",
            batch_size=32,
            lr=lr,
            opt=opt,
            epochs=5,
            pin_memory=False,
            use_amp=True
        )
        print("Dataset : MNIST, Model : ResNet-18")
        print(f"Batch Size : 32, Learning Rate : {lr}, Optimizer : {opt}, Epochs : 5, PIN_MEMORY = False")
        print(f"Test Accuracy : {acc:.2f}%")

/tmp/ipython-input-1142887438.py:95: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/tmp/ipython-input-1142887438.py:122: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


Epoch 1 average loss: 0.3911
Epoch 2 average loss: 0.0611
Epoch 3 average loss: 0.0343
Epoch 4 average loss: 0.0212
Epoch 5 average loss: 0.0122
Dataset : MNIST, Model : ResNet-18
Batch Size : 32, Learning Rate : 0.001, Optimizer : sgd, Epochs : 5, PIN_MEMORY = False
Test Accuracy : 99.03%
Epoch 1 average loss: 0.1233
Epoch 2 average loss: 0.0527
Epoch 3 average loss: 0.0389
Epoch 4 average loss: 0.0323
Epoch 5 average loss: 0.0285
Dataset : MNIST, Model : ResNet-18
Batch Size : 32, Learning Rate : 0.001, Optimizer : adam, Epochs : 5, PIN_MEMORY = False
Test Accuracy : 99.18%
Epoch 1 average loss: 1.5850
Epoch 2 average loss: 0.6769
Epoch 3 average loss: 0.3378
Epoch 4 average loss: 0.2170
Epoch 5 average loss: 0.1618
Dataset : MNIST, Model : ResNet-18
Batch Size : 32, Learning Rate : 0.0001, Optimizer : sgd, Epochs : 5, PIN_MEMORY = False
Test Accuracy : 96.70%
Epoch 1 average loss: 0.1422
Epoch 2 average loss: 0.0384
Epoch 3 average loss: 0.0260
Epoch 4 average loss: 0.0192
Epoch 5 a

MNIST - ResNet-50 - Batch size (16)

In [ ]:
for lr in [0.001, 0.0001]:
  for opt in ["sgd", "adam"]:
        acc = train_model(
            dataset_name="mnist",
            model_name="resnet50",
            batch_size=16,
            lr=lr,
            opt=opt,
            epochs=5,
            pin_memory=True,
            use_amp=True
        )
        print("Dataset : MNIST, Model : ResNet-50")
        print(f"Batch Size : 16, Learning Rate : {lr}, Optimizer : {opt}, Epochs : 5, PIN_MEMORY = True")
        print(f"Test Accuracy : {acc:.2f}%")

/tmp/ipython-input-1142887438.py:95: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/tmp/ipython-input-1142887438.py:122: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


Epoch 1 average loss: 0.3602
Epoch 2 average loss: 0.0602
Epoch 3 average loss: 0.0340
Epoch 4 average loss: 0.0213
Epoch 5 average loss: 0.0122
Dataset : MNIST, Model : ResNet-50
Batch Size : 16, Learning Rate : 0.001, Optimizer : sgd, Epochs : 5, PIN_MEMORY = True
Test Accuracy : 99.13%
Epoch 1 average loss: 0.2313
Epoch 2 average loss: 0.1016
Epoch 3 average loss: 0.0721
Epoch 4 average loss: 0.0568
Epoch 5 average loss: 0.0444
Dataset : MNIST, Model : ResNet-50
Batch Size : 16, Learning Rate : 0.001, Optimizer : adam, Epochs : 5, PIN_MEMORY = True
Test Accuracy : 98.93%
Epoch 1 average loss: 1.4485
Epoch 2 average loss: 0.4446
Epoch 3 average loss: 0.2118
Epoch 4 average loss: 0.1426
Epoch 5 average loss: 0.1044
Dataset : MNIST, Model : ResNet-50
Batch Size : 16, Learning Rate : 0.0001, Optimizer : sgd, Epochs : 5, PIN_MEMORY = True
Test Accuracy : 97.63%
Epoch 1 average loss: 0.2611
Epoch 2 average loss: 0.0757
Epoch 3 average loss: 0.0549
Epoch 4 average loss: 0.0437
Epoch 5 aver

MNIST - ResNet-50 - Batch size (32)

In [ ]:
for lr in [0.001, 0.0001]:
  for opt in ["sgd", "adam"]:
        acc = train_model(
            dataset_name="mnist",
            model_name="resnet50",
            batch_size=32,
            lr=lr,
            opt=opt,
            epochs=5,
            pin_memory=False,
            use_amp=True
        )
        print("Dataset : MNIST, Model : ResNet-50")
        print(f"Batch Size : 32, Learning Rate : {lr}, Optimizer : {opt}, Epochs : 5, PIN_MEMORY = False")
        print(f"Test Accuracy : {acc:.2f}%")

/tmp/ipython-input-1142887438.py:95: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/tmp/ipython-input-1142887438.py:122: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


Epoch 1 average loss: 0.5656
Epoch 2 average loss: 0.0748
Epoch 3 average loss: 0.0353
Epoch 4 average loss: 0.0182
Epoch 5 average loss: 0.0089
Dataset : MNIST, Model : ResNet-50
Batch Size : 32, Learning Rate : 0.001, Optimizer : sgd, Epochs : 5, PIN_MEMORY = False
Test Accuracy : 98.93%
Epoch 1 average loss: 0.2084
Epoch 2 average loss: 0.0807
Epoch 3 average loss: 0.0640
Epoch 4 average loss: 0.0574
Epoch 5 average loss: 0.0525
Dataset : MNIST, Model : ResNet-50
Batch Size : 32, Learning Rate : 0.001, Optimizer : adam, Epochs : 5, PIN_MEMORY = False
Test Accuracy : 98.51%
Epoch 1 average loss: 1.7385
Epoch 2 average loss: 0.9473
Epoch 3 average loss: 0.4997
Epoch 4 average loss: 0.3115
Epoch 5 average loss: 0.2197
Dataset : MNIST, Model : ResNet-50
Batch Size : 32, Learning Rate : 0.0001, Optimizer : sgd, Epochs : 5, PIN_MEMORY = False
Test Accuracy : 94.67%
Epoch 1 average loss: 0.2964
Epoch 2 average loss: 0.0631
Epoch 3 average loss: 0.0458
Epoch 4 average loss: 0.0365
Epoch 5 a

FashionMNIST - ResNet-18 - Batch size (16)

In [ ]:
for lr in [0.001, 0.0001]:
  for opt in ["sgd", "adam"]:
        acc = train_model(
            dataset_name="fashion",
            model_name="resnet18",
            batch_size=16,
            lr=lr,
            opt=opt,
            epochs=10,
            pin_memory=True,
            use_amp=True
        )
        print("Dataset : FashionMNIST, Model : ResNet-18")
        print(f"Batch Size : 16, Learning Rate : {lr}, Optimizer : {opt}, Epochs : 10, PIN_MEMORY = True")
        print(f"Test Accuracy : {acc:.2f}%")

/tmp/ipython-input-1142887438.py:95: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/tmp/ipython-input-1142887438.py:122: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


Epoch 1 average loss: 0.5208
Epoch 2 average loss: 0.2894
Epoch 3 average loss: 0.2223
Epoch 4 average loss: 0.1728
Epoch 5 average loss: 0.1315
Epoch 6 average loss: 0.0981
Epoch 7 average loss: 0.0762
Epoch 8 average loss: 0.0560
Epoch 9 average loss: 0.0383
Epoch 10 average loss: 0.0336
Dataset : FashionMNIST, Model : ResNet-18
Batch Size : 16, Learning Rate : 0.001, Optimizer : sgd, Epochs : 10, PIN_MEMORY = True
Test Accuracy : 91.50%
Epoch 1 average loss: 0.4380
Epoch 2 average loss: 0.2782
Epoch 3 average loss: 0.2302
Epoch 4 average loss: 0.1922
Epoch 5 average loss: 0.1553
Epoch 6 average loss: 0.1247
Epoch 7 average loss: 0.0931
Epoch 8 average loss: 0.0699
Epoch 9 average loss: 0.0537
Epoch 10 average loss: 0.0419
Dataset : FashionMNIST, Model : ResNet-18
Batch Size : 16, Learning Rate : 0.001, Optimizer : adam, Epochs : 10, PIN_MEMORY = True
Test Accuracy : 91.89%
Epoch 1 average loss: 1.1658
Epoch 2 average loss: 0.5763
Epoch 3 average loss: 0.4448
Epoch 4 average loss: 0.

FashionMNIST - ResNet-18 - Batch size (32)

In [ ]:
for lr in [0.001, 0.0001]:
  for opt in ["sgd", "adam"]:
        acc = train_model(
            dataset_name="fashion",
            model_name="resnet18",
            batch_size=32,
            lr=lr,
            opt=opt,
            epochs=5,
            pin_memory=False,
            use_amp=True
        )
        print("Dataset : FashionMNIST, Model : ResNet-18")
        print(f"Batch Size : 32, Learning Rate : {lr}, Optimizer : {opt}, Epochs : 5, PIN_MEMORY = False")
        print(f"Test Accuracy : {acc:.2f}%")

/tmp/ipython-input-1142887438.py:95: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/tmp/ipython-input-1142887438.py:122: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


Epoch 1 average loss: 0.6142
Epoch 2 average loss: 0.3159
Epoch 3 average loss: 0.2394
Epoch 4 average loss: 0.1829
Epoch 5 average loss: 0.1421
Dataset : FashionMNIST, Model : ResNet-18
Batch Size : 32, Learning Rate : 0.001, Optimizer : sgd, Epochs : 5, PIN_MEMORY = False
Test Accuracy : 90.39%
Epoch 1 average loss: 0.4189
Epoch 2 average loss: 0.2666
Epoch 3 average loss: 0.2222
Epoch 4 average loss: 0.1897
Epoch 5 average loss: 0.1599
Dataset : FashionMNIST, Model : ResNet-18
Batch Size : 32, Learning Rate : 0.001, Optimizer : adam, Epochs : 5, PIN_MEMORY = False
Test Accuracy : 92.22%
Epoch 1 average loss: 1.5108
Epoch 2 average loss: 0.7699
Epoch 3 average loss: 0.5992
Epoch 4 average loss: 0.5092
Epoch 5 average loss: 0.4493
Dataset : FashionMNIST, Model : ResNet-18
Batch Size : 32, Learning Rate : 0.0001, Optimizer : sgd, Epochs : 5, PIN_MEMORY = False
Test Accuracy : 84.40%
Epoch 1 average loss: 0.4242
Epoch 2 average loss: 0.2415
Epoch 3 average loss: 0.1740
Epoch 4 average l

FashionMNIST - ResNet-50 - Batch size (16)

In [ ]:
for lr in [0.001, 0.0001]:
  for opt in ["sgd", "adam"]:
        acc = train_model(
            dataset_name="fashion",
            model_name="resnet50",
            batch_size=16,
            lr=lr,
            opt=opt,
            epochs=5,
            pin_memory=False,
            use_amp=True
        )
        print("Dataset : FashionMNIST, Model : ResNet-50")
        print(f"Batch Size : 16, Learning Rate : {lr}, Optimizer : {opt}, Epochs : 5, PIN_MEMORY = False")
        print(f"Test Accuracy : {acc:.2f}%")

/tmp/ipython-input-1142887438.py:95: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/tmp/ipython-input-1142887438.py:122: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


Epoch 1 average loss: 0.7805
Epoch 2 average loss: 0.4032
Epoch 3 average loss: 0.3233
Epoch 4 average loss: 0.2669
Epoch 5 average loss: 0.2257
Dataset : FashionMNIST, Model : ResNet-50
Batch Size : 16, Learning Rate : 0.001, Optimizer : sgd, Epochs : 5, PIN_MEMORY = False
Test Accuracy : 89.89%
Epoch 1 average loss: 0.5223
Epoch 2 average loss: 0.3224
Epoch 3 average loss: 0.2815
Epoch 4 average loss: 0.2432
Epoch 5 average loss: 0.2136
Dataset : FashionMNIST, Model : ResNet-50
Batch Size : 16, Learning Rate : 0.001, Optimizer : adam, Epochs : 5, PIN_MEMORY = False
Test Accuracy : 89.84%
Epoch 1 average loss: 1.4849
Epoch 2 average loss: 0.7836
Epoch 3 average loss: 0.6077
Epoch 4 average loss: 0.5161
Epoch 5 average loss: 0.4581
Dataset : FashionMNIST, Model : ResNet-50
Batch Size : 16, Learning Rate : 0.0001, Optimizer : sgd, Epochs : 5, PIN_MEMORY = False
Test Accuracy : 84.32%
Epoch 1 average loss: 0.6366
Epoch 2 average loss: 0.3746
Epoch 3 average loss: 0.2967
Epoch 4 average l

FashionMNIST - ResNet-50 - Batch size (32)

In [ ]:
for lr in [0.001, 0.0001]:
  for opt in ["sgd", "adam"]:
        acc = train_model(
            dataset_name="fashion",
            model_name="resnet50",
            batch_size=32,
            lr=lr,
            opt=opt,
            epochs=5,
            pin_memory=False,
            use_amp=True
        )
        print("Dataset : FashionMNIST, Model : ResNet-50")
        print(f"Batch Size : 32, Learning Rate : {lr}, Optimizer : {opt}, Epochs : 5, PIN_MEMORY = False")
        print(f"Test Accuracy : {acc:.2f}%")

/tmp/ipython-input-1142887438.py:95: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/tmp/ipython-input-1142887438.py:122: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


Epoch 1 average loss: 0.8883
Epoch 2 average loss: 0.4412
Epoch 3 average loss: 0.3568
Epoch 4 average loss: 0.2875
Epoch 5 average loss: 0.2434
Dataset : FashionMNIST, Model : ResNet-50
Batch Size : 32, Learning Rate : 0.001, Optimizer : sgd, Epochs : 5, PIN_MEMORY = False
Test Accuracy : 88.18%
Epoch 1 average loss: 0.5270
Epoch 2 average loss: 0.3055
Epoch 3 average loss: 0.2695
Epoch 4 average loss: 0.2276
Epoch 5 average loss: 0.2050
Dataset : FashionMNIST, Model : ResNet-50
Batch Size : 32, Learning Rate : 0.001, Optimizer : adam, Epochs : 5, PIN_MEMORY = False
Test Accuracy : 92.03%
Epoch 1 average loss: 1.8789
Epoch 2 average loss: 1.0783
Epoch 3 average loss: 0.8019
Epoch 4 average loss: 0.6697
Epoch 5 average loss: 0.5882
Dataset : FashionMNIST, Model : ResNet-50
Batch Size : 32, Learning Rate : 0.0001, Optimizer : sgd, Epochs : 5, PIN_MEMORY = False
Test Accuracy : 79.97%
Epoch 1 average loss: 0.6429
Epoch 2 average loss: 0.3762
Epoch 3 average loss: 0.2975
Epoch 4 average l

# **Q1(b)**

In [2]:
import time
import numpy as np
import torch
import torchvision
import torchvision.transforms as transforms

from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

In [3]:
def load_dataset(name, train_samples=10000, test_samples=2000):
    transform = transforms.ToTensor()

    if name == "MNIST":
        trainset = torchvision.datasets.MNIST(
            root='./data', train=True, download=True, transform=transform)
        testset = torchvision.datasets.MNIST(
            root='./data', train=False, download=True, transform=transform)

    elif name == "FashionMNIST":
        trainset = torchvision.datasets.FashionMNIST(
            root='./data', train=True, download=True, transform=transform)
        testset = torchvision.datasets.FashionMNIST(
            root='./data', train=False, download=True, transform=transform)

    else:
        raise ValueError("Invalid dataset")

    X_train = trainset.data[:train_samples].reshape(train_samples, -1).numpy()
    y_train = trainset.targets[:train_samples].numpy()

    X_test = testset.data[:test_samples].reshape(test_samples, -1).numpy()
    y_test = testset.targets[:test_samples].numpy()

    return X_train, y_train, X_test, y_test

def train_svm(X_train, y_train, X_test, y_test, kernel, C=1.0, degree=3, gamma='scale'):
    model = SVC(kernel=kernel, C=C, degree=degree, gamma=gamma)

    start = time.time()
    model.fit(X_train, y_train)
    end = time.time()

    train_time_ms = (end - start) * 1000

    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred) * 100

    return accuracy, train_time_ms

In [4]:
configs = [
    ("rbf", 1.0, None),
    ("rbf", 10.0, None),
    ("poly", 1.0, 2),
    ("poly", 1.0, 3),
    ("poly", 10.0, 2),
    ("poly", 10.0, 3),
]

for dataset in ["MNIST", "FashionMNIST"]:
    X_train, y_train, X_test, y_test = load_dataset(dataset)
    for kernel, C, degree in configs:
        acc, time_ms = train_svm(
            X_train, y_train, X_test, y_test,
            kernel=kernel, C=C, degree=degree if degree else 3
        )

        print(f"Dataset: {dataset} , Kernel={kernel}, C={C}, Degree={degree} ->"
              f"Accuracy={acc:.2f}%, Time={time_ms:.2f} ms")

100%|██████████| 9.91M/9.91M [00:01<00:00, 4.97MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 160kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.52MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.62MB/s]


Dataset: MNIST , Kernel=rbf, C=1.0, Degree=None ->Accuracy=94.45%, Time=10285.61 ms
Dataset: MNIST , Kernel=rbf, C=10.0, Degree=None ->Accuracy=95.55%, Time=8977.22 ms
Dataset: MNIST , Kernel=poly, C=1.0, Degree=2 ->Accuracy=94.15%, Time=8220.28 ms
Dataset: MNIST , Kernel=poly, C=1.0, Degree=3 ->Accuracy=93.40%, Time=9914.45 ms
Dataset: MNIST , Kernel=poly, C=10.0, Degree=2 ->Accuracy=95.20%, Time=6378.44 ms
Dataset: MNIST , Kernel=poly, C=10.0, Degree=3 ->Accuracy=94.10%, Time=8375.44 ms


100%|██████████| 26.4M/26.4M [00:03<00:00, 6.72MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 152kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 2.83MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 21.4MB/s]


Dataset: FashionMNIST , Kernel=rbf, C=1.0, Degree=None ->Accuracy=86.00%, Time=9880.03 ms
Dataset: FashionMNIST , Kernel=rbf, C=10.0, Degree=None ->Accuracy=87.35%, Time=10376.90 ms
Dataset: FashionMNIST , Kernel=poly, C=1.0, Degree=2 ->Accuracy=84.75%, Time=8799.31 ms
Dataset: FashionMNIST , Kernel=poly, C=1.0, Degree=3 ->Accuracy=82.80%, Time=19842.46 ms
Dataset: FashionMNIST , Kernel=poly, C=10.0, Degree=2 ->Accuracy=86.80%, Time=8276.69 ms
Dataset: FashionMNIST , Kernel=poly, C=10.0, Degree=3 ->Accuracy=85.75%, Time=8479.50 ms
